*0.2 Math / ML basics*

# Gradient descent

**The situation.** Finance wants a latency budget: how many milliseconds does a request cost per output token? You have 500 logged requests with token counts and latencies. You need the line that fits them best — and the way to find it is the same procedure that trains every neural network, including the one you are calling.

**Gradient descent.** Start with a guess for the parameters (slope, intercept). Measure how wrong the guess is (the *loss*). Compute which direction each parameter should move to reduce the loss (the *gradient*). Take a small step that way (the *learning rate*). Repeat. PyTorch computes the gradients for you; you write the loss and the loop.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The data.** Latencies that really are about 12 ms per token plus a 300 ms base, with noise — the notebook makes them so the answer can be checked.

In [2]:
import torch

torch.manual_seed(0)
tokens = torch.randint(20, 800, (500, 1)).float()
latency_ms = (
    300 + 12 * tokens + torch.randn(500, 1) * 40
)  # the truth we hope to recover: 300 + 12 × tokens

# Two parameters to learn, starting from zero. requires_grad tells PyTorch to track them.
slope = torch.zeros(1, requires_grad=True)
intercept = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.SGD([slope, intercept], lr=0.05)

tokens_scaled = (
    tokens / 800
)  # keep inputs around 0–1 so one learning rate works for both parameters
for step in range(1, 2001):
    prediction = intercept + slope * tokens_scaled
    loss = torch.mean((prediction - latency_ms) ** 2)  # mean squared error: how wrong, on average
    optimizer.zero_grad()
    loss.backward()  # PyTorch computes d(loss)/d(slope) and d(loss)/d(intercept)
    optimizer.step()  # move each parameter a little against its gradient
    if step in (1, 10, 100, 1000, 2000):
        print(
            
                f"step {step:>5}  loss {loss.item():>12.1f}  ms/token {slope.item() / 800:>6.2f}  "
                f"base {intercept.item():>6.1f} ms"
            
        )

ms_per_token = slope.item() / 800
assert abs(ms_per_token - 12) < 1 and abs(intercept.item() - 300) < 30

step     1  loss   35175204.0  ms/token   0.44  base  524.3 ms
step    10  loss    6543156.0  ms/token   2.77  base 2949.6 ms
step   100  loss    1221990.0  ms/token   7.29  base 2376.8 ms
step  1000  loss       1806.5  ms/token  11.99  base  306.9 ms
step  2000  loss       1796.8  ms/token  12.00  base  301.1 ms


**Reading the output.** The loss falls step after step and the parameters settle near 12 ms/token and a 300 ms base — the truth the data was made from. Nobody told the program the answer; it walked downhill on the loss until it found it.

```
loss
 │ ●
 │   ●
 │     ●
 │        ●  ●
 │              ●   ●    ●    ●    ●     ← flat: the minimum
 └────────────────────────────────────── steps
```

**The rule to remember.** Guess → measure the loss → gradient → small step → repeat. Training a 70-billion-parameter model is this loop, with a bigger loss function and more GPUs.

| Use it when | Don't when | Instead use |
|---|---|---|
| any model with parameters to learn: fine-tuning, classifiers, this regression | the problem has a closed-form solution and few parameters | `numpy.linalg.lstsq` / `sklearn.linear_model.LinearRegression` |

**Watch out**
- Learning rate is the knob that matters most. Too high: the loss jumps around or explodes to `nan`. Too low: nothing moves. Scale inputs first (as above) so one rate works.
- `optimizer.zero_grad()` every step. Forgetting it accumulates gradients — the classic silent bug.
- Plain SGD is for teaching; `torch.optim.AdamW` is what training code actually uses.